# Trimet Data Project

For our project, we decided to work with Trimet data as it sounded interesting and there seem and their developer portal have very good documentation for accessing data.

There were 2 static data GIS (Geospatial Data) and GFTS Static (General Transit Feed Specification).  The GIS basically provides an outline of the Transit map such as stop locations, route boundaries, garage locations and GFTS Static mainly contains text files like `stops.txt` `stop_times.txt` which shows how trimet vehicles are planned to move.

The main interesting we found that had lots of data was GFTS Realtime.  This logs information on Trimet vehicles in realtime giving back information like location using longitude and latitude, whether the vehicle is delayed or early, what its next stop and previous stop is and sometimes even an approximate of how full the vehicle is.

We worked with this api and built a collector in python to collect data on vehicles every minute.  We ran the collector program as much as we can as you cannot have too much data. The data we recieved from the api is formated in json which we converted into a csv.  In this conversion to csv, we only selected specific columns which sounded interesting in order to minimize the size of the csv file.



## Data Processing

Trimet provided documentation for an api called [/vehicles](https://developer.trimet.org/ws_docs/vehicle_locations_ws.shtml) showing information on what data is collected for a vehicle in real time

Example Json data for 1 vehicle.  This is collected every minute.

*Every minute, it logs about 400 vehicles during the day and almost none at night.*

```
      {
        "routeColor": "61A744",
        "expires": 1760993085102,
        "signMessage": "FX2 To Gresham",
        "serviceDate": 1760943600000,
        "loadPercentage": 15, #
        "latitude": 45.503744191408664, #
        "nextStopSeq": 8,
        "source": "vm",
        "type": "bus",
        "blockID": 248,
        "signMessageLong": "FX2 Division To Gresham",
        "lastLocID": 3397,
        "nextLocID": 13732,
        "locationInScheduleDay": 49133, #
        "routeSubType": "BRT",
        "newTrip": false,
        "longitude": -122.66883380836836, #
        "direction": 0,
        "inCongestion": false, #
        "routeNumber": 2, #
        "bearing": 46,
        "garage": "POWELL",
        "tripID": "15644087", #
        "delay": -112, #
        "extraBlockID": null,
        "messageCode": 100,
        "lastStopSeq": 7,
        "vehicleID": 4531,
        "time": 1760992845102, #
        "offRoute": false #
      },

```

## Notable information from data documentation

* type - type of vehicle (bus or rail)
* longitude, latitude - location of vehicle
* inCongestion (Experimental) - *true* if vehicle not moving in traffic
* loadPercentage (Experimental) - estimate how many people riding
* vehicle_id - identifier
* time - time when position was recorded
* delay - positive means late, negative means early

* bearing: which direction vehicle is going.
* nextLocID - the previous stop
  
* routeNumber - Route number the vehicle is in (FX2, FX9)
* tripID - the planned trip from *trips.txt*
* garage - 

*Some fields from exmple data not shown because it was optional.  Some buses might record it.*

## Creating DataFrame from csv

In [ ]:
import pandas as pd

In [ ]:
df = pd.read_csv('data/vehicle_positions_full_raw.csv')

## Understanding the data

In [13]:
df.head(2)

,bearing,blockID,collection_timestamp,delay,direction,expires,extraBlockID,garage,inCongestion,lastLocID,...,routeNumber,routeSubType,serviceDate,signMessage,signMessageLong,source,time,tripID,type,vehicleID
0,200.0,243,1760817789,-1132,0,1760818026912,NaN,POWELL,False,7642.0,...,2,BRT,1760770800000,FX2 To Gresham,FX2 Division To Gresham,vm,1760817786912,15643905,bus,4530
1,269.0,248,1760817789,-499,1,1760818022926,NaN,POWELL,False,14241.0,...,2,BRT,1760770800000,FX2 To Portland,FX2 Division To Portland,vm,1760817782926,15644364,bus,4528


In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 727199 entries, 0 to 727198
Data columns (total 31 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   bearing                727184 non-null  float64
 1   blockID                727199 non-null  int64  
 2   collection_timestamp   727199 non-null  int64  
 3   delay                  727199 non-null  int64  
 4   direction              727199 non-null  int64  
 5   expires                727199 non-null  int64  
 6   extraBlockID           1611 non-null    float64
 7   garage                 725588 non-null  object 
 8   inCongestion           562449 non-null  object 
 9   lastLocID              601971 non-null  float64
 10  lastStopSeq            601971 non-null  float64
 11  latitude               727199 non-null  float64
 12  loadPercentage         491051 non-null  float64
 13  locationInScheduleDay  727199 non-null  int64  
 14  longitude              727199 non-nu

### Non obvious columns

Some columns are not straightforward and will need more research to get better understanding.

#### routeSubType

In [11]:
# Check what RouteSubtype column is:
df['routeSubType'].unique()

array(['BRT', 'Bus', 'Light Rail', 'Shuttle'], dtype=object)

This shows that `routeSubType` gives more specification to vehicle info.  A BRT is "Bus Rapid Transit" and a "Bus" is the smaller old buses both are categorized as `bus` for the `type` column.

#### offRoute

Perhaps we can see if vehicles ever go off route, if not we can drop the entire column

In [15]:
df['offRoute'].value_counts()

offRoute
False    722640
True       4559
Name: count, dtype: int64

A small portion goes off route which will be interesting to anlyze later on.

#### Extra Block ID


In [ ]:
df['extraBlock']

### Non important columns

* vm - internal tool by trimet to label vehicle
* 

### Checking for outliers

In [26]:
df.describe()

,bearing,blockID,collection_timestamp,delay,direction,expires,extraBlockID,lastLocID,lastStopSeq,latitude,...,locationInScheduleDay,longitude,messageCode,nextLocID,nextStopSeq,routeNumber,serviceDate,time,tripID,vehicleID
count,727184.000000,727199.000000,7.271990e+05,727199.000000,727199.000000,7.271990e+05,1611.000000,601971.000000,601971.000000,727199.000000,...,727199.000000,727199.000000,727199.000000,727199.000000,727199.000000,727199.000000,7.271990e+05,7.271990e+05,7.271990e+05,727199.000000
mean,175.633082,5168.212412,1.760903e+09,-128.287474,0.498400,1.760903e+12,740.400993,7563.460567,29.581302,45.507922,...,51229.364086,-122.664693,613.279281,7899.231099,24.523740,64.997089,1.760851e+12,1.760903e+12,1.562486e+07,2905.597924
std,102.343308,3249.810505,5.177268e+04,375.320404,0.499998,5.178153e+07,27.388769,4042.059391,24.164961,0.048849,...,18010.849297,0.127264,324.258516,4132.484296,24.947662,60.402103,6.127143e+07,5.177813e+07,6.911754e+05,1439.332417
min,0.000000,134.000000,1.760818e+09,-8595.000000,0.000000,1.760818e+12,701.000000,2.000000,1.000000,45.284999,...,11784.000000,-123.115567,13.000000,2.000000,1.000000,1.000000,1.760684e+12,1.760817e+12,9.007170e+05,103.000000
25%,90.000000,1902.000000,1.760850e+09,-157.000000,0.000000,1.760850e+12,702.000000,4439.000000,11.000000,45.490587,...,35950.000000,-122.715628,316.000000,4651.000000,3.000000,19.000000,1.760771e+12,1.760850e+12,1.565053e+07,3024.000000
50%,180.000000,5473.000000,1.760906e+09,-30.000000,0.000000,1.760906e+12,736.000000,8045.000000,24.000000,45.517125,...,51277.000000,-122.662312,637.000000,8345.000000,17.000000,57.000000,1.760857e+12,1.760906e+12,1.565755e+07,3420.000000
75%,270.000000,8743.000000,1.760938e+09,0.000000,1.000000,1.760938e+12,767.000000,10028.000000,42.000000,45.530573,...,65024.000000,-122.578744,863.000000,10611.000000,38.000000,88.000000,1.760857e+12,1.760938e+12,1.566456e+07,3935.000000
max,359.000000,9767.000000,1.760986e+09,8893.000000,1.000000,1.760989e+12,770.000000,14661.000000,133.000000,45.639116,...,97800.000000,-122.330200,1094.000000,14661.000000,134.000000,293.000000,1.760944e+12,1.760986e+12,1.567294e+07,4531.000000


In [25]:
print(f'max latitude position {df['latitude'].max()}')
print(f'min latitude position {df['latitude'].min()}')
print(f'max longitude position {df['longitude'].max()}')
print(f'min longitude position {df['longitude'].min()}')

max latitude position 45.63911567610055
min latitude position 45.284998982742216
max longitude position -122.3302
min longitude position -123.115567
